# CER-SCM: independent replication (30 assets)

An independent run of the same pipeline in which all 30 tickers across 5 sectors (including commodities) downloaded successfully. Code and outputs exactly as run.

> **Note.** As in the main run, the Sortino values in the recorded outputs come from an earlier, invalid formula and should be ignored. See the README.

## 1. Dependencies

In [2]:
# FIX: pin gcastle to avoid NotearsNonlinear API breakage between versions
!pip install "gcastle==1.0.3" networkx scipy scikit-learn statsmodels yfinance -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use('Agg')           # headless - safe on server
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings, gc, os, time, json, copy
warnings.filterwarnings('ignore')

from castle.algorithms import PC
try:
    from castle.algorithms import NotearsNonlinear
    _NOTEARS_OK = True
except ImportError:
    _NOTEARS_OK = False
    print("  [WARN] NotearsNonlinear not available - will use PC fallback for neural SCM")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler   # mixed precision

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# FIX: RTX 3090 optimisations
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False   # faster, still reproducible per seed

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'AMP     : enabled (float16 forward, float32 master weights)')
print('OK  All imports successful')



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


2026-06-12 12:42:00,347 - /home/nmit/anaconda3/lib/python3.13/site-packages/castle/backend/__init__.py[line:36] - INFO: You can use `os.environ['CASTLE_BACKEND'] = backend` to set the backend(`pytorch` or `mindspore`).
2026-06-12 12:42:00,399 - /home/nmit/anaconda3/lib/python3.13/site-packages/castle/algorithms/__init__.py[line:36] - INFO: You are using ``pytorch`` as the backend.


PyTorch : 2.7.1+cu118
Device  : cuda
GPU     : NVIDIA GeForce RTX 3090
VRAM    : 25.3 GB
AMP     : enabled (float16 forward, float32 master weights)
OK  All imports successful


## 2. Configuration - RTX 3090 (24 GB VRAM)

In [3]:
ASSET_UNIVERSE = {
    'US_Tech'    : ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'AMZN'],
    'US_Finance' : ['JPM', 'GS', 'BAC', 'MS', 'BLK', 'C'],
    'Asian_Mkts' : ['TSM', '9984.T', '005930.KS', '7203.T', '6758.T', '000660.KS'],
    'EU_Mkts'    : ['ASML', 'SAP', 'SIE.DE', 'BAS.DE', 'AIR.PA', 'DTE.DE'],
    'Commodities': ['GLD', 'SLV', 'USO', 'DBA', 'PDBC', 'IAU'],
}
STOCK_TICKERS = [t for s in ASSET_UNIVERSE.values() for t in s]  # 30 tickers

TRAIN_START = '2018-01-01';  TRAIN_END = '2021-12-31'
VAL_START   = '2022-01-01';  VAL_END   = '2022-12-31'
TEST_START  = '2023-01-01';  TEST_END  = '2024-12-31'

N_TRIALS    = 10
N_EPISODES  = 100
INITIAL_BAL = 100_000
TRANS_COST  = 0.001

CF_RATIO      = 0.5
CQL_ALPHA     = 0.01
N_CF_ACTIONS  = 2
CF_BETA       = 0.9      # epistemic discount on counterfactual rewards

# RTX 3090: 24 GB VRAM - generous buffer & batch
BUFFER_CAP  = 200_000
BATCH_SIZE  = 256

TARGET_UPDATE_FREQ = 10   # episodes between hard target updates (DQN)

CKPT_DIR = 'journal_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'Assets   : {len(STOCK_TICKERS)} tickers across {len(ASSET_UNIVERSE)} sectors')
print(f'Trials   : {N_TRIALS} x {N_EPISODES} episodes')
print(f'Buffer   : {BUFFER_CAP:,}  |  Batch: {BATCH_SIZE}')
print(f'Device   : {DEVICE}')
print(f'CF beta  : {CF_BETA}  |  CQL alpha: {CQL_ALPHA}')


Assets   : 30 tickers across 5 sectors
Trials   : 10 x 100 episodes
Buffer   : 200,000  |  Batch: 256
Device   : cuda
CF beta  : 0.9  |  CQL alpha: 0.01


## 3. Data Loading & Feature Engineering (12 features)

In [4]:
class TradingDataLoader:
    FEATURE_COLS = [
        'returns', 'volatility', 'volume_ratio',
        'momentum_5', 'momentum_10', 'momentum_20',
        'rsi', 'bb_position', 'macd_diff',
        'atr_ratio', 'obv_signal', 'price_vs_sma50',
    ]

    def __init__(self, ticker, start, end):
        self.ticker = ticker
        self.start  = start
        self.end    = end

    def load(self):
        try:
            raw = yf.download(self.ticker, start=self.start, end=self.end,
                              progress=False, auto_adjust=True)
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.droplevel(1)
            if len(raw) < 150:
                raise ValueError(f'Only {len(raw)} rows - insufficient history')
        except Exception as e:
            print(f'  WARN  {self.ticker}: {e}')
            return None

        df    = raw.copy()
        close = df['Close'].squeeze()
        high  = df['High'].squeeze()
        low   = df['Low'].squeeze()
        vol   = df['Volume'].squeeze()

        df['returns']      = close.pct_change()
        df['volatility']   = df['returns'].rolling(20).std()
        df['volume_ma']    = vol.rolling(20).mean()
        df['volume_ratio'] = vol / (df['volume_ma'] + 1e-9)
        df['momentum_5']   = close.pct_change(5)
        df['momentum_10']  = close.pct_change(10)
        df['momentum_20']  = close.pct_change(20)

        delta = close.diff()
        gain  = delta.where(delta > 0, 0).rolling(14).mean()
        loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
        df['rsi'] = (100 - 100 / (1 + gain / (loss + 1e-9))) / 100.0

        bb_mid = close.rolling(20).mean()
        bb_std = close.rolling(20).std()
        df['bb_position'] = (close - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-9)

        exp1 = close.ewm(span=12).mean()
        exp2 = close.ewm(span=26).mean()
        macd = exp1 - exp2
        df['macd_diff'] = (macd - macd.ewm(span=9).mean()) / (close + 1e-9)

        tr = pd.concat([high - low,
                        (high - close.shift()).abs(),
                        (low  - close.shift()).abs()], axis=1).max(axis=1)
        df['atr_ratio'] = tr.rolling(14).mean() / (close + 1e-9)

        obv = (np.sign(df['returns']) * vol).cumsum()
        df['obv_signal'] = obv.pct_change(10)

        sma50 = close.rolling(50).mean()
        df['price_vs_sma50'] = (close - sma50) / (sma50 + 1e-9)

        df = df[self.FEATURE_COLS + ['Close', 'Volume']].dropna()
        for col in self.FEATURE_COLS:
            lo = df[col].quantile(0.001)
            hi = df[col].quantile(0.999)
            df[col] = df[col].clip(lo, hi)

        return df

STATE_DIM  = len(TradingDataLoader.FEATURE_COLS)   # 12
STATE_COLS = TradingDataLoader.FEATURE_COLS
print(f'OK  TradingDataLoader ready  ({STATE_DIM} features)')


OK  TradingDataLoader ready  (12 features)


## 4. Causal Structure Learning - PC and NOTEARS-MLP

In [5]:
CAUSAL_VARS = [
    'returns', 'volatility', 'volume_ratio',
    'momentum_5', 'rsi', 'bb_position', 'macd_diff',
    'atr_ratio', 'obv_signal', 'price_vs_sma50',
]

def learn_causal_structure(features_df, alpha=0.05, method='pc'):
    """
    Learn causal graph from financial time series.
    Returns (causal_matrix, var_names, scaled_data_df)
    """
    dm = features_df[CAUSAL_VARS].copy()
    dm['action']       = np.sign(features_df['momentum_5'])
    dm['returns_lag1'] = features_df['returns'].shift(1)
    dm = dm.dropna()

    var_names    = list(dm.columns)
    scaler       = StandardScaler()
    dm_scaled    = scaler.fit_transform(dm.values)
    dm_scaled_df = pd.DataFrame(dm_scaled, columns=var_names)

    if method == 'notears' and _NOTEARS_OK:
        try:
            model = NotearsNonlinear()
            model.learn(dm_scaled)
            graph   = model.causal_matrix
            n_edges = int(np.sum(graph != 0))
            print(f'    [NOTEARS-MLP] {len(var_names)} nodes, {n_edges} edges')
            return graph, var_names, dm_scaled_df
        except Exception as e:
            print(f'    [NOTEARS failed -> PC fallback]: {e}')

    # PC algorithm (default / fallback)
    pc    = PC(alpha=alpha)
    pc.learn(dm_scaled)
    graph = pc.causal_matrix
    print(f'    [PC alpha={alpha}] {len(var_names)} nodes, {int(np.sum(graph!=0))} edges')
    return graph, var_names, dm_scaled_df

print('OK  Causal structure learner ready (PC + NOTEARS-MLP with fallback)')


OK  Causal structure learner ready (PC + NOTEARS-MLP with fallback)


## 5. Structural Causal Model - Linear and Neural SEMs

Pearl's 3-step do-calculus:
1. **Abduction** - infer exogenous noise `U_i` from observations
2. **Action** - intervene `do(action = a')`
3. **Prediction** - propagate through causal graph in topological order


In [6]:
class StructuralCausalModel:
    def __init__(self, causal_graph, var_names, mode='linear'):
        self.graph       = causal_graph
        self.var_names   = var_names
        self.n           = len(var_names)
        self.mode        = mode
        self.models      = {}
        self._r2_scores  = {}
        self._topo_order = None

    def fit(self, data_matrix: pd.DataFrame):
        self._topo_order = self._topological_sort()
        for i, var in enumerate(self.var_names):
            parent_idx = [j for j in range(self.n) if self.graph[j, i] != 0]
            if not parent_idx:
                self.models[var] = {
                    'type': 'exogenous',
                    'mean': float(data_matrix.iloc[:, i].mean()),
                    'std' : float(data_matrix.iloc[:, i].std()) + 1e-8,
                }
                self._r2_scores[var] = None
            else:
                parent_names = [self.var_names[p] for p in parent_idx]
                X = data_matrix[parent_names].values
                y = data_matrix[var].values
                reg = (MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu',
                                    max_iter=500, random_state=42,
                                    early_stopping=True, validation_fraction=0.1)
                       if self.mode == 'neural' else LinearRegression())
                reg.fit(X, y)
                y_pred    = reg.predict(X)
                residuals = y - y_pred
                self.models[var] = {
                    'type'         : 'endogenous',
                    'parent_names' : parent_names,
                    'reg'          : reg,
                    'residual_std' : float(np.std(residuals)) + 1e-8,
                }
                self._r2_scores[var] = float(r2_score(y, y_pred))

    def fit_report(self):
        endo = {k: v for k, v in self._r2_scores.items() if v is not None}
        if not endo:
            return {}
        return {'mean_r2': float(np.mean(list(endo.values()))),
                'min_r2' : float(min(endo.values())),
                'per_var': endo}

    def generate_counterfactual_state(self, observed_row: dict,
                                      intervention_var: str,
                                      intervention_value: float) -> dict:
        # Step 1 - Abduction: recover exogenous noise
        noise = {}
        for var in self.var_names:
            m = self.models.get(var)
            if m is None:
                noise[var] = 0.
                continue
            if m['type'] == 'exogenous':
                noise[var] = observed_row.get(var, m['mean'])
            else:
                pv   = np.array([observed_row.get(p, 0.) for p in m['parent_names']]).reshape(1, -1)
                pred = float(m['reg'].predict(pv)[0])
                noise[var] = observed_row.get(var, pred) - pred

        # Step 2 - Action: apply intervention
        cf = dict(observed_row)
        cf[intervention_var] = intervention_value   # FIX: was 'intervention_val' (typo)

        # Step 3 - Prediction: propagate forward in topological order
        int_idx     = self.var_names.index(intervention_var)
        descendants = self._get_descendants(int_idx)
        for var in self._topo_order:
            if var == intervention_var:
                continue
            if self.var_names.index(var) not in descendants:
                continue
            m = self.models.get(var)
            if m and m['type'] == 'endogenous':
                pv   = np.array([cf.get(p, 0.) for p in m['parent_names']]).reshape(1, -1)
                pred = float(m['reg'].predict(pv)[0])
                cf[var] = pred + noise.get(var, 0.)
        return cf

    def _topological_sort(self):
        in_deg = {v: 0 for v in self.var_names}
        for i in range(self.n):
            for j in range(self.n):
                if self.graph[i, j] != 0:
                    in_deg[self.var_names[j]] += 1
        queue = [v for v, d in in_deg.items() if d == 0]
        order = []
        while queue:
            node = queue.pop(0)
            order.append(node)
            idx = self.var_names.index(node)
            for j in range(self.n):
                if self.graph[idx, j] != 0:
                    in_deg[self.var_names[j]] -= 1
                    if in_deg[self.var_names[j]] == 0:
                        queue.append(self.var_names[j])
        # FIX: fall back gracefully if graph has cycles (can happen with NOTEARS)
        return order if len(order) == self.n else list(self.var_names)

    def _get_descendants(self, node_idx):
        visited, queue = set(), [node_idx]
        while queue:
            cur = queue.pop(0)
            for j in range(self.n):
                if self.graph[cur, j] != 0 and j not in visited:
                    visited.add(j)
                    queue.append(j)
        return visited

print('OK  StructuralCausalModel (linear + neural, do-calculus) ready')


OK  StructuralCausalModel (linear + neural, do-calculus) ready


## 6. Trading Environment

In [7]:
class TradingEnvironment:
    def __init__(self, data, initial_balance=INITIAL_BAL, tc=TRANS_COST):

        # Convert once to fast NumPy array
        if isinstance(data, pd.DataFrame):
            self.data = data[STATE_COLS + ['Close']].values.astype(np.float32)
        else:
            self.data = data.astype(np.float32)

        self.init_bal = initial_balance
        self.tc       = tc
        self.reset()

    def reset(self):
        self.step_idx = 0
        self.balance  = self.init_bal
        self.shares   = 0.0
        self.pv       = self.init_bal
        self.history  = []
        return self._state()

    def _state(self):

        if self.step_idx >= len(self.data):
            return np.zeros(STATE_DIM, dtype=np.float32)

        # Fast NumPy indexing instead of pandas iloc
        s = self.data[self.step_idx][:STATE_DIM]

        return np.nan_to_num(
            s,
            nan=0.0,
            posinf=1.0,
            neginf=-1.0
        ).astype(np.float32)

    def step(self, action):

        if self.step_idx >= len(self.data) - 1:
            return self._state(), 0.0, True

        # Close price is last column
        price      = float(self.data[self.step_idx][-1])
        next_price = float(self.data[self.step_idx + 1][-1])

        # Buy
        if action == 2 and self.balance > 0:
            self.shares += self.balance / (price * (1 + self.tc))
            self.balance = 0.0

        # Sell
        elif action == 0 and self.shares > 0:
            self.balance += self.shares * price * (1 - self.tc)
            self.shares = 0.0

        old_pv  = self.pv
        self.pv = self.balance + self.shares * next_price

        reward = (self.pv - old_pv) / (old_pv + 1e-9)

        self.history.append({
            'step': self.step_idx,
            'action': action,
            'pv': self.pv,
            'reward': reward
        })

        self.step_idx += 1

        done = (self.step_idx >= len(self.data) - 1)

        return self._state(), reward, done

    def metrics(self):

        if not self.history:
            return {
                'total_return': 0.0,
                'sharpe_ratio': 0.0,
                'sortino_ratio': 0.0,
                'calmar_ratio': 0.0,
                'max_drawdown': 0.0,
                'win_rate': 0.0,
                'turnover': 0.0,
                'final_value': 0.0
            }

        h = pd.DataFrame(self.history)

        rets = h['reward'].values
        pv   = h['pv'].values

        tr = (self.pv - self.init_bal) / self.init_bal

        sh = (
            np.mean(rets) /
            (np.std(rets) + 1e-9)
        ) * np.sqrt(252)

        neg = rets[rets < 0]

        so = (
            np.mean(rets) /
            (np.std(neg) + 1e-9)
        ) * np.sqrt(252) if len(neg) > 0 else sh

        peak = np.maximum.accumulate(pv)

        mdd = float(
            np.min((pv - peak) / (peak + 1e-9))
        )

        ann = tr * 252 / max(len(rets), 1)

        cal = ann / (abs(mdd) + 1e-9)

        return {
            'total_return': tr,
            'sharpe_ratio': sh,
            'sortino_ratio': so,
            'calmar_ratio': cal,
            'max_drawdown': mdd,
            'win_rate': float(np.mean(rets > 0)),
            'turnover': float(np.mean(h['action'] != 1)),
            'final_value': self.pv
        }

print(f'OK  TradingEnvironment ready  (state_dim={STATE_DIM})')

OK  TradingEnvironment ready  (state_dim=12)


## 7. Networks & Replay Buffers

In [8]:
class DuelingQNet(nn.Module):
    def __init__(self, state_dim=STATE_DIM, action_dim=3, hidden=(512, 256)):
        super().__init__()
        layers, in_d = [], state_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(0.1)]
            in_d = h
        self.shared = nn.Sequential(*layers)
        self.value  = nn.Sequential(nn.Linear(in_d, 128), nn.ReLU(), nn.Linear(128, 1))
        self.adv    = nn.Sequential(nn.Linear(in_d, 128), nn.ReLU(), nn.Linear(128, action_dim))
        self.float()

    def forward(self, x):
        h = self.shared(x)
        v = self.value(h)
        a = self.adv(h)
        return v + (a - a.mean(dim=1, keepdim=True))


class SACActorDiscrete(nn.Module):
    def __init__(self, state_dim=STATE_DIM, action_dim=3, hidden=(512, 256)):
        super().__init__()
        layers, in_d = [], state_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU()]
            in_d = h
        layers.append(nn.Linear(in_d, action_dim))
        self.net = nn.Sequential(*layers)
        self.float()

    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

    def get_action(self, x):
        probs    = self.forward(x)
        dist     = torch.distributions.Categorical(probs)
        action   = dist.sample()
        log_prob = torch.log(probs + 1e-8)
        entropy  = -(probs * log_prob).sum(dim=-1, keepdim=True)
        return action, probs, log_prob, entropy


class SACCriticDiscrete(nn.Module):
    def __init__(self, state_dim=STATE_DIM, action_dim=3, hidden=(512, 256)):
        super().__init__()
        def mlp():
            layers, in_d = [], state_dim
            for h in hidden:
                layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU()]
                in_d = h
            layers.append(nn.Linear(in_d, action_dim))
            return nn.Sequential(*layers)
        self.q1 = mlp()
        self.q2 = mlp()
        self.float()

    def forward(self, x):
        return self.q1(x), self.q2(x)


# FIX: replace=True when buffer smaller than batch to avoid crash on early training
class PrioritizedReplayBuffer:
    def __init__(self, capacity=BUFFER_CAP, alpha=0.6,
                 beta_start=0.4, beta_end=1.0, beta_steps=100_000):
        self.capacity  = capacity
        self.alpha     = alpha
        self.beta      = beta_start
        self.beta_end  = beta_end
        self.beta_inc  = (beta_end - beta_start) / beta_steps
        self.buf       = []
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.ptr       = 0
        self.max_prio  = 1.0

    def push(self, *t):
        if len(self.buf) < self.capacity:
            self.buf.append(t)
        else:
            self.buf[self.ptr] = t
        self.priorities[self.ptr] = self.max_prio
        self.ptr = (self.ptr + 1) % self.capacity

    def sample(self, n):
        buf_len = len(self.buf)
        n       = min(n, buf_len)
        prios   = self.priorities[:buf_len] ** self.alpha
        probs   = prios / prios.sum()
        # FIX: use replace=True when buf smaller than requested n (early training)
        replace = (buf_len < n)
        idx     = np.random.choice(buf_len, n, replace=replace, p=probs)
        w       = (buf_len * probs[idx]) ** (-self.beta)
        w      /= w.max() + 1e-8
        self.beta = min(self.beta_end, self.beta + self.beta_inc)
        return [self.buf[i] for i in idx], idx, w

    def update_priorities(self, indices, td_errors):
        for i, e in zip(indices, td_errors):
            p = float(abs(e)) + 1e-6
            self.priorities[i] = p
            self.max_prio = max(self.max_prio, p)

    def __len__(self): return len(self.buf)


class UniformReplayBuffer:
    def __init__(self, capacity=BUFFER_CAP):
        self.capacity = capacity
        self.buf = []
        self.ptr = 0

    def push(self, *t):
        if len(self.buf) < self.capacity:
            self.buf.append(t)
        else:
            self.buf[self.ptr] = t
            self.ptr = (self.ptr + 1) % self.capacity

    def sample(self, n):
        n   = min(n, len(self.buf))
        idx = np.random.choice(len(self.buf), n, replace=False)
        return [self.buf[i] for i in idx]

    def __len__(self): return len(self.buf)

print('OK  DuelingQNet, SAC nets, PrioritizedReplayBuffer, UniformReplayBuffer ready')


OK  DuelingQNet, SAC nets, PrioritizedReplayBuffer, UniformReplayBuffer ready


## 8. Agents - Dueling DQN + SAC + CER Mixin

In [9]:
def batch_to_tensors(batch, device):
    s  = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32).to(device)
    a  = torch.tensor(np.array([b[1] for b in batch]), dtype=torch.long).to(device)
    r  = torch.tensor(np.array([b[2] for b in batch]), dtype=torch.float32).to(device)
    s2 = torch.tensor(np.array([b[3] for b in batch]), dtype=torch.float32).to(device)
    d  = torch.tensor(np.array([b[4] for b in batch]), dtype=torch.float32).to(device)
    return s, a, r, s2, d


# ── Dueling Double DQN (baseline) ────────────────────────────────────────────
class DuelingDQNAgent:
    def __init__(self, lr=3e-4, gamma=0.99, device=DEVICE):
        self.gamma    = gamma
        self.device   = device
        self.q_net    = DuelingQNet().to(device).float()
        self.tgt_net  = DuelingQNet().to(device).float()
        self.tgt_net.load_state_dict(self.q_net.state_dict())
        self.optim    = optim.Adam(self.q_net.parameters(), lr=lr, weight_decay=1e-5)
        self.memory   = PrioritizedReplayBuffer(BUFFER_CAP)
        self.batch    = BATCH_SIZE
        self.scaler   = GradScaler()     # AMP gradient scaler

    def act(self, state, eps=0.1):
        if np.random.random() < eps:
            return np.random.randint(3)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
            
            return int(self.q_net(s).argmax())

    def push(self, *t): self.memory.push(*t)

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        batch, idx, w = self.memory.sample(self.batch)
        return self._update(batch, idx, w)

    def _update(self, batch, idx=None, w=None):
        s, a, r, s2, d = batch_to_tensors(batch, self.device)
        n  = len(batch)
        wt = torch.FloatTensor(w[:n]).to(self.device) if w is not None              else torch.ones(n, device=self.device)

        with autocast():
            q_cur = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze()
            with torch.no_grad():
                next_a = self.q_net(s2).argmax(1)
                q_next = self.tgt_net(s2).gather(1, next_a.unsqueeze(1)).squeeze()
                q_tgt  = r + self.gamma * q_next * (1 - d)
            td_err = (q_cur - q_tgt).detach()
            loss   = (wt * F.smooth_l1_loss(q_cur, q_tgt, reduction='none')).mean()

        self.optim.zero_grad()
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.optim)
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.)
        self.scaler.step(self.optim)
        self.scaler.update()

        if idx is not None:
            self.memory.update_priorities(idx, td_err.cpu().numpy())
        return float(loss)

    # FIX: hard update only - called from training loop every TARGET_UPDATE_FREQ episodes
    def sync_target(self):
        self.tgt_net.load_state_dict(self.q_net.state_dict())


# ── Discrete SAC ─────────────────────────────────────────────────────────────
class DiscreteSACAgent:
    def __init__(self, lr=3e-4, gamma=0.99, alpha=0.2,
                 auto_alpha=True, target_entropy_ratio=0.98, device=DEVICE):
        self.gamma  = gamma
        self.device = device
        self.actor = SACActorDiscrete().to(device).float()
        self.critic = SACCriticDiscrete().to(device).float()
        self.critic_target = SACCriticDiscrete().to(device).float()
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.actor_opt  = optim.Adam(self.actor.parameters(),  lr=lr)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=lr)
        self.memory     = UniformReplayBuffer(BUFFER_CAP)
        self.batch      = BATCH_SIZE
        self.scaler     = GradScaler()

        self.auto_alpha = auto_alpha
        if auto_alpha:
            self.target_entropy = -np.log(1.0 / 3) * target_entropy_ratio
            self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
            self.alpha     = self.log_alpha.exp().item()
            self.alpha_opt = optim.Adam([self.log_alpha], lr=lr)
        else:
            self.alpha = alpha

    def act(self, state, eps=0.):
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
            
            return int(self.actor(s).argmax())

    def push(self, *t): self.memory.push(*t)

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        batch = self.memory.sample(self.batch)
        s, a, r, s2, d = batch_to_tensors(batch, self.device)

        with autocast():
            with torch.no_grad():
                _, probs2, log_probs2, _ = self.actor.get_action(s2)
                q1_next, q2_next = self.critic_target(s2)
                q_next   = torch.min(q1_next, q2_next)
                v_next   = (probs2 * (q_next - self.alpha * log_probs2)).sum(1)
                q_target = r + self.gamma * v_next * (1 - d)

            q1, q2 = self.critic(s)
            q1_a   = q1.gather(1, a.unsqueeze(1)).squeeze()
            q2_a   = q2.gather(1, a.unsqueeze(1)).squeeze()
            c_loss = F.mse_loss(q1_a, q_target) + F.mse_loss(q2_a, q_target)

        self.critic_opt.zero_grad()
        self.scaler.scale(c_loss).backward()
        self.scaler.unscale_(self.critic_opt)
        nn.utils.clip_grad_norm_(self.critic.parameters(), 10.)
        self.scaler.step(self.critic_opt)
        self.scaler.update()

        with autocast():
            _, probs, log_probs, _ = self.actor.get_action(s)
            q1_pi, q2_pi = self.critic(s)
            q_pi   = torch.min(q1_pi, q2_pi)
            a_loss = (probs * (self.alpha * log_probs - q_pi)).sum(1).mean()

        self.actor_opt.zero_grad()
        self.scaler.scale(a_loss).backward()
        self.scaler.unscale_(self.actor_opt)
        nn.utils.clip_grad_norm_(self.actor.parameters(), 10.)
        self.scaler.step(self.actor_opt)
        self.scaler.update()

        if self.auto_alpha:
            with autocast():
                alpha_loss = -(
                        self.log_alpha *
                        (log_probs.detach() + self.target_entropy) *
                        probs.detach()
                    ).sum(1).mean()
            self.alpha_opt.zero_grad()
            self.scaler.scale(alpha_loss).backward()
            self.scaler.step(self.alpha_opt)
            self.scaler.update()
            self.alpha = self.log_alpha.exp().item()

        for p, tp in zip(self.critic.parameters(), self.critic_target.parameters()):
            tp.data.copy_(0.005 * p.data + 0.995 * tp.data)

        return float(c_loss)

    def sync_target(self): pass  # SAC uses soft updates inside train_step


# ── CER Mixin ────────────────────────────────────────────────────────────────
class CERMixin:
    def init_cer(self, scm, var_names, cql_alpha=CQL_ALPHA):
        self.scm           = scm
        self.var_names     = var_names
        self.cql_alpha     = cql_alpha
        self.cf_memory     = UniformReplayBuffer(BUFFER_CAP)
        self.state_var_map = {v: i for i, v in enumerate(STATE_COLS) if v in var_names}

    def _state_to_obs(self, state):
        obs = {v: float(state[i]) for v, i in self.state_var_map.items()}
        for v in self.var_names:
            if v not in obs:
                obs[v] = 0.
        return obs

    def _obs_to_state(self, obs, original_state):
        cf = original_state.copy()
        for v, i in self.state_var_map.items():
            if v in obs:
                cf[i] = float(np.nan_to_num(obs[v], nan=0., posinf=1., neginf=-1.))
        return cf

    @staticmethod
    def _a2scm(a): return {0: -1., 1: 0., 2: 1.}[a]

    def generate_and_store_counterfactuals(self, state, action, reward, next_state, done):
        obs = self._state_to_obs(state)
        if 'action' in self.var_names:
            obs['action'] = self._a2scm(action)
        for alt_a in [a for a in range(3) if a != action][:N_CF_ACTIONS]:
            cf_obs   = self.scm.generate_counterfactual_state(obs, 'action', self._a2scm(alt_a))
            cf_state = self._obs_to_state(cf_obs, state)
            cf_r_raw = cf_obs.get('returns', reward)
            cf_r     = float(np.nan_to_num(cf_r_raw, nan=0., posinf=0.1, neginf=-0.1)) * CF_BETA
            self.cf_memory.push(cf_state, alt_a, cf_r, next_state, done)


# ── CausalDQN: Dueling DQN + PER + CER + Variance-Penalized CQL ──────────────
class CausalDQNAgent(DuelingDQNAgent, CERMixin):
    def __init__(self, scm, var_names, **kwargs):
        super().__init__(**kwargs)
        self.init_cer(scm, var_names)

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        real_n          = self.batch // 2
        cf_n            = self.batch - real_n
        real_b, idx, w  = self.memory.sample(real_n)

        if len(self.cf_memory) >= cf_n:
            cf_b  = self.cf_memory.sample(cf_n)
            mixed = real_b + cf_b
            # FIX: w only covers real_n samples; extend with ones for CF
            w_full = np.concatenate([w, np.ones(cf_n)])
        else:
            mixed  = real_b
            w_full = w

        s, a, r, s2, d = batch_to_tensors(mixed, self.device)
        n  = len(mixed)
        wt = torch.FloatTensor(w_full[:n]).to(self.device)

        with autocast():
            q_cur = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze()
            with torch.no_grad():
                next_a = self.q_net(s2).argmax(1)
                q_all  = self.tgt_net(s2)
                q_next = q_all.gather(1, next_a.unsqueeze(1)).squeeze()
                q_std  = q_all.std(dim=1)
                # Variance-Penalized CQL target
                q_cons = q_next - self.cql_alpha * q_std
                q_tgt  = r + self.gamma * q_cons * (1 - d)
            td_err = (q_cur - q_tgt).detach()
            loss   = (wt * F.smooth_l1_loss(q_cur, q_tgt, reduction='none')).mean()

        self.optim.zero_grad()
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.optim)
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.)
        self.scaler.step(self.optim)
        self.scaler.update()

        # FIX: only update PER priorities for the real_n real transitions
        self.memory.update_priorities(idx, td_err[:real_n].cpu().numpy())
        return float(loss)


# ── CausalSAC: SAC + CER ──────────────────────────────────────────────────────
class CausalSACAgent(DiscreteSACAgent, CERMixin):
    def __init__(self, scm, var_names, **kwargs):
        super().__init__(**kwargs)
        self.init_cer(scm, var_names)

    def train_step(self):
        # Mix real + CF transitions, then call SAC update on the mixed batch
        real_n = self.batch // 2
        cf_n   = self.batch - real_n
        if len(self.memory) < real_n:
            return 0.

        real_b = self.memory.sample(real_n)
        if len(self.cf_memory) >= cf_n:
            mixed = real_b + self.cf_memory.sample(cf_n)
        else:
            mixed = real_b

        s, a, r, s2, d = batch_to_tensors(mixed, self.device)

        with autocast():
            with torch.no_grad():
                _, probs2, log_probs2, _ = self.actor.get_action(s2)
                q1_next, q2_next = self.critic_target(s2)
                q_next   = torch.min(q1_next, q2_next)
                v_next   = (probs2 * (q_next - self.alpha * log_probs2)).sum(1)
                q_target = r + self.gamma * v_next * (1 - d)

            q1, q2 = self.critic(s)
            q1_a   = q1.gather(1, a.unsqueeze(1)).squeeze()
            q2_a   = q2.gather(1, a.unsqueeze(1)).squeeze()
            c_loss = F.mse_loss(q1_a, q_target) + F.mse_loss(q2_a, q_target)

        self.critic_opt.zero_grad()
        self.scaler.scale(c_loss).backward()
        self.scaler.unscale_(self.critic_opt)
        nn.utils.clip_grad_norm_(self.critic.parameters(), 10.)
        self.scaler.step(self.critic_opt)
        self.scaler.update()

        with autocast():
            _, probs, log_probs, _ = self.actor.get_action(s)
            q1_pi, q2_pi = self.critic(s)
            q_pi   = torch.min(q1_pi, q2_pi)
            a_loss = (probs * (self.alpha * log_probs - q_pi)).sum(1).mean()

        self.actor_opt.zero_grad()
        self.scaler.scale(a_loss).backward()
        self.scaler.unscale_(self.actor_opt)
        nn.utils.clip_grad_norm_(self.actor.parameters(), 10.)
        self.scaler.step(self.actor_opt)
        self.scaler.update()

        if self.auto_alpha:
            with autocast():
                alpha_loss = -(
                        self.log_alpha *
                        (log_probs.detach() + self.target_entropy) *
                        probs.detach()
                    ).sum(1).mean()
            self.alpha_opt.zero_grad()
            self.scaler.scale(alpha_loss).backward()
            self.scaler.step(self.alpha_opt)
            self.scaler.update()
            self.alpha = self.log_alpha.exp().item()

        for p, tp in zip(self.critic.parameters(), self.critic_target.parameters()):
            tp.data.copy_(0.005 * p.data + 0.995 * tp.data)

        return float(c_loss)

print('OK  DuelingDQN, SAC, CausalDQN (AMP), CausalSAC (AMP) agents ready')


OK  DuelingDQN, SAC, CausalDQN (AMP), CausalSAC (AMP) agents ready


## 9. Ablation Variants

In [10]:
# FIX: RandomCFDQNAgent - correct MRO; call DuelingDQNAgent.__init__ explicitly
class RandomCFDQNAgent(DuelingDQNAgent):
    """Ablation: random Gaussian perturbation instead of SCM counterfactuals."""
    def __init__(self, noise_scale=0.05, **kwargs):
        DuelingDQNAgent.__init__(self, **kwargs)
        self.cf_memory   = UniformReplayBuffer(BUFFER_CAP)
        self.noise_scale = noise_scale
        self.cql_alpha   = CQL_ALPHA   # keep CQL for fair comparison

    def generate_and_store_counterfactuals(self, state, action, reward, next_state, done):
        for alt_a in [a for a in range(3) if a != action][:N_CF_ACTIONS]:
            cf_s = (state + np.random.normal(0, self.noise_scale,
                                             state.shape)).astype(np.float32)
            cf_r = float(reward + np.random.normal(0, self.noise_scale))
            self.cf_memory.push(cf_s, alt_a, cf_r, next_state, done)

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        real_n          = self.batch // 2
        real_b, idx, w  = self.memory.sample(real_n)
        if len(self.cf_memory) >= self.batch - real_n:
            mixed  = real_b + self.cf_memory.sample(self.batch - real_n)
            w_full = np.concatenate([w, np.ones(self.batch - real_n)])
        else:
            mixed  = real_b
            w_full = w
        return self._update(mixed, idx, w_full)


class NoCQLDQNAgent(CausalDQNAgent):
    """Ablation: CER without CQL conservative penalty (cql_alpha = 0)."""
    def __init__(self, scm, var_names, **kwargs):
        super().__init__(scm, var_names, **kwargs)
        self.cql_alpha = 0.0


class NoPERDQNAgent(CausalDQNAgent):
    """Ablation: CER + CQL, uniform replay (no PER)."""
    def __init__(self, scm, var_names, **kwargs):
        super().__init__(scm, var_names, **kwargs)
        self.memory = UniformReplayBuffer(BUFFER_CAP)   # replace PER with uniform

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        real_n = self.batch // 2
        real_b = self.memory.sample(real_n)
        if len(self.cf_memory) >= self.batch - real_n:
            mixed = real_b + self.cf_memory.sample(self.batch - real_n)
        else:
            mixed = real_b
        # No IS weights for uniform buffer - pass w=None
        return self._update(mixed, idx=None, w=None)

print('OK  Ablation agents ready: RandomCF, NoCQL, NoPER')


OK  Ablation agents ready: RandomCF, NoCQL, NoPER


## 10. Validation-Based Early Stopping & Training Loop

In [11]:
def evaluate_on_val(agent, val_data, n_episodes=3):
    sharpes = []
    for _ in range(n_episodes):
        env   = TradingEnvironment(val_data)
        state = env.reset()
        done  = False
        while not done:
            action = agent.act(state, eps=0.)
            state, _, done = env.step(action)
        sharpes.append(env.metrics()['sharpe_ratio'])
    return float(np.mean(sharpes))


def train_agent(agent, train_data, val_data, n_episodes=N_EPISODES,
                use_cer=False, patience=30, verbose=False):
    eps, eps_min, eps_decay = 0.8, 0.02, 0.99
    best_val_sharpe = -np.inf
    best_state_dict = None
    no_improve      = 0
    eval_every      = 10

    for ep in range(n_episodes):
        env   = TradingEnvironment(train_data)
        state = env.reset()
        done  = False

        while not done:
            action = agent.act(state, eps)
            ns, reward, done = env.step(action)
            agent.push(state, action, reward, ns, float(done))

            if use_cer and hasattr(agent, 'generate_and_store_counterfactuals'):
                agent.generate_and_store_counterfactuals(state, action, reward, ns, done)

            agent.train_step()
            state = ns

        eps = max(eps_min, eps * eps_decay)

        # FIX: target sync every TARGET_UPDATE_FREQ episodes, NOT every step
        if hasattr(agent, 'sync_target') and (ep + 1) % TARGET_UPDATE_FREQ == 0:
            agent.sync_target()

        # Validation check
        if val_data is not None and (ep + 1) % eval_every == 0:
            val_sharpe = evaluate_on_val(agent, val_data)
            if val_sharpe > best_val_sharpe:
                best_val_sharpe = val_sharpe
                if hasattr(agent, 'q_net'):
                    best_state_dict = copy.deepcopy(agent.q_net.state_dict())
                elif hasattr(agent, 'actor'):
                    best_state_dict = copy.deepcopy(agent.actor.state_dict())
                no_improve = 0
            else:
                no_improve += eval_every
                if no_improve >= patience:
                    if verbose:
                        print(f'      Early stop ep {ep+1}  best_val_sharpe={best_val_sharpe:.3f}')
                    break

    if best_state_dict is not None:
        if hasattr(agent, 'q_net'):
            agent.q_net.load_state_dict(best_state_dict)
        elif hasattr(agent, 'actor'):
            agent.actor.load_state_dict(best_state_dict)


def evaluate_agent(agent, test_data):
    env   = TradingEnvironment(test_data)
    state = env.reset()
    done  = False
    while not done:
        action = agent.act(state, eps=0.)
        state, _, done = env.step(action)
    return env.metrics()


def buy_and_hold(data):
    p   = data['Close'].values
    sh  = INITIAL_BAL / (p[0] * (1 + TRANS_COST))
    fv  = sh * p[-1] * (1 - TRANS_COST)
    r   = pd.Series(p).pct_change().dropna().values
    sh2 = np.mean(r) / (np.std(r) + 1e-9) * np.sqrt(252)
    neg = r[r < 0]
    so  = np.mean(r) / (np.std(neg) + 1e-9) * np.sqrt(252) if len(neg) > 0 else sh2
    cum = (1 + pd.Series(r)).cumprod()
    mdd = float((cum / cum.cummax() - 1).min())
    ann = ((fv / INITIAL_BAL) ** (252 / max(len(r), 1))) - 1
    return {'total_return': (fv - INITIAL_BAL) / INITIAL_BAL,
            'sharpe_ratio': sh2, 'sortino_ratio': so,
            'calmar_ratio': ann / (abs(mdd) + 1e-9),
            'max_drawdown': mdd, 'win_rate': float(np.mean(r > 0)),
            'final_value': fv}


def rule_strategy(data, rule='momentum'):
    env   = TradingEnvironment(data)
    state = env.reset()
    for i in range(len(data) - 1):
        if rule == 'momentum':
            m      = float(data.iloc[i]['momentum_10'])
            action = 2 if m > 0.02 else (0 if m < -0.02 else 1)
        else:
            rsi    = float(data.iloc[i]['rsi'])
            action = 2 if rsi < 0.30 else (0 if rsi > 0.70 else 1)
        state, _, done = env.step(action)
        if done: break
    return env.metrics()

print('OK  Training loop + early stopping + baselines ready')


OK  Training loop + early stopping + baselines ready


## 11. Checkpointing Utilities

In [12]:
METHODS_RL     = ['causal_dqn', 'causal_sac', 'dueling_dqn', 'discrete_sac',
                  'random_cf', 'no_cql', 'no_per']
METHODS_STATIC = ['buy_hold', 'momentum', 'mean_reversion']
ALL_METHODS    = METHODS_RL + METHODS_STATIC

def ckpt_path(ticker):
    return os.path.join(CKPT_DIR, f'{ticker.replace(".", "_")}.json')

def _strip_history(v):
    """Remove 'history' DataFrames before JSON serialisation."""
    if isinstance(v, list):
        return [{k2: vv for k2, vv in r.items() if k2 != 'history'} for r in v]
    if isinstance(v, dict):
        return {k2: vv for k2, vv in v.items() if k2 != 'history'}
    return v

def save_ckpt(ticker, data):
    out = {m: _strip_history(v) for m, v in data.items()}
    with open(ckpt_path(ticker), 'w') as f:
        json.dump(out, f, indent=2)

def load_ckpt(ticker):
    p = ckpt_path(ticker)
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return None

def already_done(ticker):
    ck = load_ckpt(ticker)
    if ck is None:
        return False
    for m in METHODS_RL:
        if len(ck.get(m, [])) < N_TRIALS:
            return False
    return True

existing = [t for t in STOCK_TICKERS if already_done(t)]
print(f'OK  Checkpointing ready  |  dir: {CKPT_DIR}/')
print(f'    Already completed: {existing if existing else "none"}')


OK  Checkpointing ready  |  dir: journal_checkpoints/
    Already completed: ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'AMZN', 'JPM', 'GS', 'BAC', 'MS', 'BLK', 'C', 'TSM', '9984.T', '005930.KS', '7203.T', '6758.T', '000660.KS', 'ASML', 'SAP', 'SIE.DE', 'BAS.DE', 'AIR.PA', 'DTE.DE', 'GLD', 'SLV', 'USO', 'DBA', 'PDBC', 'IAU']


In [13]:
!nvidia-smi

Fri Jun 12 12:42:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:41:00.0  On |                  N/A |
|  0%   35C    P8             17W /  370W |     340MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 12. Main Experiment Loop

> **Runtime:** several days on an RTX 3090.
> Checkpoints are saved per ticker, so a run can be interrupted and resumed.


In [14]:
results          = {}
experiment_start = time.time()

for ticker in STOCK_TICKERS:
    print(f'\n{"="*65}')
    print(f'  {ticker}  [{STOCK_TICKERS.index(ticker)+1}/{len(STOCK_TICKERS)}]')
    print(f'{"="*65}')

    if already_done(ticker):
        print('  OK  Already complete - loading checkpoint')
        results[ticker] = load_ckpt(ticker)
        continue

    # Load data
    train_data = TradingDataLoader(ticker, TRAIN_START, TRAIN_END).load()
    val_data   = TradingDataLoader(ticker, VAL_START,   VAL_END).load()
    test_data  = TradingDataLoader(ticker, TEST_START,  TEST_END).load()

    train_np = train_data.values.astype(np.float32)
    val_np   = val_data.values.astype(np.float32)
    test_np  = test_data.values.astype(np.float32)
    
    if train_data is None or test_data is None:
        print('  Skipping - data unavailable')
        continue
    if val_data is None:
        val_data = train_data.iloc[-int(len(train_data) * 0.15):]
        print(f'  Val fallback: last 15% of train ({len(val_data)} rows)')

    print(f'  Train:{len(train_data)}  Val:{len(val_data)}  Test:{len(test_data)}')

    # Causal structure (once per ticker)
    print('  [1/5] PC causal graph...')
    t_cg = time.time()
    cg_pc, vn, dm = learn_causal_structure(train_data, method='pc')
    print(f'        PC done in {time.time()-t_cg:.1f}s')

    print('  [2/5] NOTEARS-MLP causal graph...')
    t_nt = time.time()
    cg_notears, _, _ = learn_causal_structure(train_data, method='notears')
    print(f'        NOTEARS done in {time.time()-t_nt:.1f}s')

    # Fit SCMs
    print('  [3/5] Fitting SCMs...')
    scm_linear = StructuralCausalModel(cg_pc,      vn, mode='linear')
    scm_linear.fit(dm)
    scm_neural = StructuralCausalModel(cg_notears, vn, mode='neural')
    scm_neural.fit(dm)
    rep  = scm_linear.fit_report()
    rep2 = scm_neural.fit_report()
    if rep:
        print(f'    Linear SCM  R2 mean={rep["mean_r2"]:.3f}  min={rep["min_r2"]:.3f}')
    if rep2:
        print(f'    Neural SCM  R2 mean={rep2["mean_r2"]:.3f}  min={rep2["min_r2"]:.3f}')

    # Deterministic baselines (no randomness, run once)
    print('  [4/5] Static baselines...')
    res = {m: [] for m in METHODS_RL}
    res['buy_hold']       = buy_and_hold(test_data)
    res['momentum']       = rule_strategy(test_data, 'momentum')
    res['mean_reversion'] = rule_strategy(test_data, 'mean_reversion')
    print(f'    B&H {res["buy_hold"]["total_return"]*100:+.1f}%  '
          f'Mom {res["momentum"]["total_return"]*100:+.1f}%  '
          f'MR {res["mean_reversion"]["total_return"]*100:+.1f}%')

    # RL Trials
    print(f'  [5/5] {N_TRIALS} trials x {N_EPISODES} episodes...')
    t0 = time.time()

    for trial in range(N_TRIALS):
        set_seed(100 + trial)

        ag = CausalDQNAgent(scm_linear, vn)
        train_agent(ag, train_data, val_data, use_cer=True)
        res['causal_dqn'].append(evaluate_agent(ag, test_data))
        del ag;  gc.collect();  torch.cuda.empty_cache()

        ag = CausalSACAgent(scm_neural, vn)
        train_agent(ag, train_data, val_data, use_cer=True)
        res['causal_sac'].append(evaluate_agent(ag, test_data))
        del ag;  gc.collect();  torch.cuda.empty_cache()

        ag = DuelingDQNAgent()
        train_agent(ag, train_data, val_data, use_cer=False)
        res['dueling_dqn'].append(evaluate_agent(ag, test_data))
        del ag;  gc.collect();  torch.cuda.empty_cache()

        ag = DiscreteSACAgent()
        train_agent(ag, train_data, val_data, use_cer=False)
        res['discrete_sac'].append(evaluate_agent(ag, test_data))
        del ag;  gc.collect();  torch.cuda.empty_cache()

        ag = RandomCFDQNAgent()
        train_agent(ag, train_data, val_data, use_cer=True)
        res['random_cf'].append(evaluate_agent(ag, test_data))
        del ag;  gc.collect();  torch.cuda.empty_cache()

        ag = NoCQLDQNAgent(scm_linear, vn)
        train_agent(ag, train_data, val_data, use_cer=True)
        res['no_cql'].append(evaluate_agent(ag, test_data))
        del ag;  gc.collect();  torch.cuda.empty_cache()

        ag = NoPERDQNAgent(scm_linear, vn)
        train_agent(ag, train_data, val_data, use_cer=True)
        res['no_per'].append(evaluate_agent(ag, test_data))
        del ag;  gc.collect();  torch.cuda.empty_cache()

        if (trial + 1) % 5 == 0:
            cd = np.mean([r['total_return'] for r in res['causal_dqn']])
            cs = np.mean([r['total_return'] for r in res['causal_sac']])
            dd = np.mean([r['total_return'] for r in res['dueling_dqn']])
            elapsed = (time.time() - t0) / 60
            print(f'    Trial {trial+1:2d}/{N_TRIALS} ({elapsed:.0f}min) | '
                  f'CausalDQN {cd*100:+.1f}%  '
                  f'CausalSAC {cs*100:+.1f}%  '
                  f'DuelingDQN {dd*100:+.1f}%')

    results[ticker] = res
    save_ckpt(ticker, res)
    print(f'  OK  {ticker} done in {(time.time()-t0)/60:.1f} min - checkpoint saved')

total_h = (time.time() - experiment_start) / 3600
print(f'\nOK  All experiments complete in {total_h:.2f} hours')



  AAPL  [1/30]
  OK  Already complete - loading checkpoint

  MSFT  [2/30]
  OK  Already complete - loading checkpoint

  NVDA  [3/30]
  OK  Already complete - loading checkpoint

  GOOGL  [4/30]
  OK  Already complete - loading checkpoint

  META  [5/30]
  OK  Already complete - loading checkpoint

  AMZN  [6/30]
  OK  Already complete - loading checkpoint

  JPM  [7/30]
  OK  Already complete - loading checkpoint

  GS  [8/30]
  OK  Already complete - loading checkpoint

  BAC  [9/30]
  OK  Already complete - loading checkpoint

  MS  [10/30]
  OK  Already complete - loading checkpoint

  BLK  [11/30]
  OK  Already complete - loading checkpoint

  C  [12/30]
  OK  Already complete - loading checkpoint

  TSM  [13/30]
  OK  Already complete - loading checkpoint

  9984.T  [14/30]
  OK  Already complete - loading checkpoint

  005930.KS  [15/30]
  OK  Already complete - loading checkpoint

  7203.T  [16/30]
  OK  Already complete - loading checkpoint

  6758.T  [17/30]
  OK  Already c

## 13. Statistical Analysis

In [16]:
valid = [t for t in STOCK_TICKERS if t in results and results[t].get('buy_hold')]
print(f'Valid tickers: {len(valid)}')

def flat_metric(method, metric):
    vals = []
    if method in METHODS_RL:
        for t in valid:
            trials = results[t].get(method, [])
            if trials:
                vals.extend([r[metric] for r in trials if metric in r])
    else:
        for t in valid:
            m = results[t].get(method)
            if isinstance(m, dict) and metric in m:
                vals.append(m[metric])
    return vals

all_ret    = {m: flat_metric(m, 'total_return')  for m in ALL_METHODS}
all_sharpe = {m: flat_metric(m, 'sharpe_ratio')  for m in ALL_METHODS}
all_sort   = {m: flat_metric(m, 'sortino_ratio') for m in METHODS_RL}
all_cal    = {m: flat_metric(m, 'calmar_ratio')  for m in METHODS_RL}


# Wilcoxon: CausalDQN vs every other method
cr      = all_ret['causal_dqn']
w_pvals = {}
c_ds    = {}
for m in [x for x in ALL_METHODS if x != 'causal_dqn']:
    comp = all_ret[m]
    n    = min(len(cr), len(comp))
    if n < 10:
        w_pvals[m] = float('nan')
        c_ds[m]    = float('nan')
        continue
    try:
        _, p = stats.wilcoxon(cr[:n], comp[:n])
    except Exception:
        _, p = stats.mannwhitneyu(cr[:n], comp[:n], alternative='two-sided')
    w_pvals[m] = p
    md_ = np.mean(cr) - np.mean(comp)
    ps_ = np.sqrt((np.var(cr) + np.var(comp)) / 2)
    c_ds[m] = md_ / (ps_ + 1e-9)

valid_keys    = [k for k, v in w_pvals.items() if not np.isnan(v)]
_, pvals_corr, _, _ = multipletests([w_pvals[k] for k in valid_keys], method='bonferroni')
corrected = dict(zip(valid_keys, pvals_corr))

# Friedman across RL methods
per_stock = {m: [np.mean([r['total_return'] for r in results[t][m]])
                 for t in valid if results[t].get(m)]
             for m in METHODS_RL}
try:
    f_stat, f_pval = stats.friedmanchisquare(*[per_stock[m] for m in METHODS_RL
                                               if len(per_stock[m]) > 0])
except Exception as e:
    f_stat, f_pval = float('nan'), float('nan')
    print(f'Friedman: {e}')

print('='*82)
print('JOURNAL STATISTICAL RESULTS')
print('='*82)
print(f'{"Method":<22}{"Mean Ret%":>10}{"+-Std":>7}{"Sharpe":>8}'
      f'{"Wilcox p":>12}{"Corr p":>10}{"d":>8}')
print('-'*82)
for m in ALL_METHODS:
    mr  = np.mean(all_ret[m]) * 100   if all_ret[m]    else float('nan')
    sr  = np.std(all_ret[m])  * 100   if all_ret[m]    else float('nan')
    sh  = np.mean(all_sharpe[m])      if all_sharpe[m] else float('nan')
    wp  = w_pvals.get(m, float('nan'))
    cp  = corrected.get(m, float('nan'))
    cd_ = c_ds.get(m, float('nan'))
    sig = '**' if cp < 0.01 else ('*' if cp < 0.05 else '')
    print(f'{m:<22}{mr:>+9.2f}%{sr:>6.2f}%{sh:>8.3f}'
          f'{wp:>11.4f}{cp:>9.4f}{sig:<2}{cd_:>+7.3f}')

print(f'\nFriedman (RL methods): chi2={f_stat:.3f}  p={f_pval:.6f}')
print('* p<0.05  ** p<0.01  (Bonferroni corrected)')


Valid tickers: 30
JOURNAL STATISTICAL RESULTS
Method                 Mean Ret%  +-Std  Sharpe    Wilcox p    Corr p       d
----------------------------------------------------------------------------------
causal_dqn               +25.59% 54.68%   0.481        nan      nan     +nan
causal_sac               +35.64% 67.26%   0.651     0.0019   0.0168*  -0.164
dueling_dqn              +28.68% 55.97%   0.552     0.0923   0.8305   -0.056
discrete_sac             +20.90% 46.50%   0.426     0.0289   0.2602   +0.092
random_cf                +14.96% 29.75%   0.314     0.0004   0.0038** +0.242
no_cql                   +30.44% 53.62%   0.546     0.0340   0.3060   -0.089
no_per                   +30.45% 53.68%   0.575     0.0199   0.1788   -0.090
buy_hold                 +78.91% 85.02%   1.204     0.0000   0.0002** -0.746
momentum                 +29.99% 57.21%   0.640     0.3184   1.0000   -0.079
mean_reversion           +27.77% 23.50%   0.976     0.0473   0.4253   -0.052

Friedman (RL methods):

## 14. Ablation Study Table

In [17]:
ABLATION_LABELS = {
    'causal_dqn'  : 'Full CER-DQN (ours)',
    'dueling_dqn' : '-CER (Dueling DQN only)',
    'random_cf'   : '-SCM (random perturbation)',
    'no_cql'      : '-CQL (no conservative penalty)',
    'no_per'      : '-PER (uniform replay)',
    'causal_sac'  : 'CER-SAC (neural SCM)',
}

abl_rows = []
for m, label in ABLATION_LABELS.items():
    rets = flat_metric(m, 'total_return')
    sh   = flat_metric(m, 'sharpe_ratio')
    so   = flat_metric(m, 'sortino_ratio')
    ca   = flat_metric(m, 'calmar_ratio')
    mdd  = flat_metric(m, 'max_drawdown')
    wr   = flat_metric(m, 'win_rate')
    abl_rows.append({
        'Variant' : label,
        'Ret%'    : f'{np.mean(rets)*100:+.2f}+-{np.std(rets)*100:.2f}' if rets else 'N/A',
        'Sharpe'  : f'{np.mean(sh):.3f}'  if sh  else 'N/A',
        'Sortino' : f'{np.mean(so):.3f}'  if so  else 'N/A',
        'Calmar'  : f'{np.mean(ca):.3f}'  if ca  else 'N/A',
        'MaxDD%'  : f'{np.mean(mdd)*100:.2f}' if mdd else 'N/A',
        'WinRate' : f'{np.mean(wr):.3f}'  if wr  else 'N/A',
    })

df_abl = pd.DataFrame(abl_rows)
print('ABLATION STUDY')
print(df_abl.to_string(index=False))
df_abl.to_csv('journal_ablation.csv', index=False)
print('\nOK  Saved: journal_ablation.csv')


ABLATION STUDY
                       Variant          Ret% Sharpe   Sortino Calmar MaxDD% WinRate
           Full CER-DQN (ours) +25.59+-54.68  0.481 11827.623  0.968 -13.72   0.224
       -CER (Dueling DQN only) +28.68+-55.97  0.552  1661.039  0.973 -14.37   0.227
    -SCM (random perturbation) +14.96+-29.75  0.314 16939.366  0.736  -7.79   0.155
-CQL (no conservative penalty) +30.44+-53.62  0.546  1668.782  0.969 -15.18   0.255
         -PER (uniform replay) +30.45+-53.68  0.575  1324.146  1.065 -15.80   0.258
          CER-SAC (neural SCM) +35.64+-67.26  0.651  6264.482  1.260 -17.06   0.273

OK  Saved: journal_ablation.csv


## 15. Per-Asset Results Table

In [18]:
rows = []
for t in valid:
    sector = next((s for s, lst in ASSET_UNIVERSE.items() if t in lst), 'Unknown')
    row    = {'Asset': t, 'Sector': sector}
    for m in METHODS_RL + METHODS_STATIC:
        if m in METHODS_RL and results[t].get(m):
            row[f'{m}_ret%']   = f'{np.mean([r["total_return"] for r in results[t][m]])*100:+.2f}'
            row[f'{m}_sharpe'] = f'{np.mean([r["sharpe_ratio"] for r in results[t][m]]):.3f}'
        elif m in METHODS_STATIC and isinstance(results[t].get(m), dict):
            row[f'{m}_ret%'] = f'{results[t][m]["total_return"]*100:+.2f}'
    rows.append(row)

df_full = pd.DataFrame(rows)
display_cols = ['Asset', 'Sector',
                'causal_dqn_ret%', 'causal_dqn_sharpe',
                'causal_sac_ret%',
                'dueling_dqn_ret%', 'discrete_sac_ret%',
                'buy_hold_ret%']
print(df_full[[c for c in display_cols if c in df_full.columns]].to_string(index=False))
df_full.to_csv('journal_full_results.csv', index=False)
print('\nOK  Saved: journal_full_results.csv')


    Asset      Sector causal_dqn_ret% causal_dqn_sharpe causal_sac_ret% dueling_dqn_ret% discrete_sac_ret% buy_hold_ret%
     AAPL     US_Tech           +5.60             0.130          +26.15           +29.08            +19.71        +65.99
     MSFT     US_Tech          +23.93             0.840          +14.13           +15.85            +14.64        +61.94
     NVDA     US_Tech         +147.70             1.093         +259.32          +205.22           +114.45       +466.67
    GOOGL     US_Tech          +13.25             0.178          +32.48           +22.45            +16.48        +99.30
     META     US_Tech          +57.31             1.006          +85.23           +70.99            +46.92       +199.53
     AMZN     US_Tech          +17.93             0.579          +41.49           +23.21            +24.24       +129.58
      JPM  US_Finance          +60.62             1.286          +69.62           +47.19            +40.64        +94.91
       GS  US_Finance          +

## 16. Publication-Quality Figures

In [19]:
plt.rcParams.update({
    'figure.dpi': 300, 'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.fontsize': 9, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
})
PAL = {
    'causal_dqn'    : '#E63946', 'causal_sac'    : '#C1121F',
    'dueling_dqn'   : '#457B9D', 'discrete_sac'  : '#1D3557',
    'random_cf'     : '#F4A261', 'no_cql'         : '#2A9D8F',
    'no_per'        : '#8338EC', 'buy_hold'       : '#A8DADC',
    'momentum'      : '#6D6875', 'mean_reversion' : '#B5838D',
}
LAB = {
    'causal_dqn'    : 'Causal DQN (Ours)', 'causal_sac'   : 'Causal SAC (Ours)',
    'dueling_dqn'   : 'Dueling DQN',       'discrete_sac' : 'Discrete SAC',
    'random_cf'     : 'Random CF',          'no_cql'       : 'CER w/o CQL',
    'no_per'        : 'CER w/o PER',        'buy_hold'     : 'Buy-Hold',
    'momentum'      : 'Momentum',           'mean_reversion': 'Mean Reversion',
}

def savefig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight')
    plt.savefig(f'{name}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'OK  Saved: {name}.pdf/.png')

# Figure 1: Violin - Return & Sharpe
plot_m = ['causal_dqn','causal_sac','dueling_dqn','discrete_sac','random_cf','buy_hold','momentum']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (metric, ylabel, src) in zip(axes, [
    ('total_return', 'Total Return', all_ret),
    ('sharpe_ratio', 'Sharpe Ratio', all_sharpe),
]):
    data_list = [src[m] for m in plot_m if src.get(m)]
    valid_m   = [m for m in plot_m if src.get(m)]
    parts = ax.violinplot(data_list, positions=range(len(valid_m)),
                          showmeans=True, showmedians=True)
    for pc, m in zip(parts['bodies'], valid_m):
        pc.set_facecolor(PAL[m]); pc.set_alpha(0.72)
    ax.set_xticks(range(len(valid_m)))
    ax.set_xticklabels([LAB[m] for m in valid_m], rotation=35, ha='right')
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3, axis='y')

axes[0].set_title('Return Distribution', fontweight='bold')
axes[1].set_title('Sharpe Distribution', fontweight='bold')
plt.suptitle(f'CER-SCM vs Baselines - {len(valid)} Assets, {N_TRIALS} Trials each',
             fontsize=12, y=1.01)
plt.tight_layout()
savefig('fig1_distributions')


OK  Saved: fig1_distributions.pdf/.png


In [20]:
# Figure 2: Ablation bars
abl_m   = list(ABLATION_LABELS.keys())
metrics = [('total_return', 'Total Return'), ('sharpe_ratio', 'Sharpe'),
           ('sortino_ratio', 'Sortino'),      ('calmar_ratio', 'Calmar')]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (met, title) in zip(axes, metrics):
    vals = [np.mean(flat_metric(m, met)) * (100 if met == 'total_return' else 1)
            for m in abl_m]
    colors = ['#E63946' if m == 'causal_dqn' else '#457B9D' for m in abl_m]
    bars = ax.bar(range(len(abl_m)), vals, color=colors, alpha=0.85)
    for bar in bars[:1]:
        bar.set_edgecolor('black'); bar.set_linewidth(2)
    ax.set_xticks(range(len(abl_m)))
    ax.set_xticklabels([ABLATION_LABELS[m] for m in abl_m], rotation=40, ha='right', fontsize=8)
    ax.axhline(0, color='gray', lw=0.6, ls='--')
    ax.set_title(title, fontweight='bold'); ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Ablation Study - Component Contributions', fontsize=13, y=1.02)
plt.tight_layout()
savefig('fig2_ablation')

# Figure 3: Risk-Return scatter
fig, ax = plt.subplots(figsize=(10, 8))
for m in ALL_METHODS:
    if not all_ret.get(m) or not all_sharpe.get(m): continue
    ax.scatter(all_sharpe[m], [r * 100 for r in all_ret[m]],
               alpha=0.25, s=20, color=PAL[m], label=None)
    ax.scatter(np.mean(all_sharpe[m]), np.mean(all_ret[m]) * 100,
               s=180, marker='*', color=PAL[m], zorder=5, label=LAB[m])
ax.axhline(0, color='gray', lw=0.8, ls='--'); ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Sharpe Ratio'); ax.set_ylabel('Total Return (%)')
ax.set_title('Risk-Return Frontier', fontweight='bold')
ax.legend(loc='upper left', framealpha=0.9); ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig('fig3_risk_return')

# Figure 4: Sector heatmap
heat_m    = ['causal_dqn','causal_sac','dueling_dqn','buy_hold']
sectors   = list(ASSET_UNIVERSE.keys())
heat_data = np.zeros((len(sectors), len(heat_m)))
for si, sec in enumerate(sectors):
    tickers_s = ASSET_UNIVERSE[sec]
    for mi, m in enumerate(heat_m):
        vals = []
        for t in tickers_s:
            if t not in valid: continue
            if m in METHODS_RL and results[t].get(m):
                vals.append(np.mean([r['total_return'] for r in results[t][m]]))
            elif m in METHODS_STATIC and isinstance(results[t].get(m), dict):
                vals.append(results[t][m]['total_return'])
        heat_data[si, mi] = np.mean(vals) * 100 if vals else 0.

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(heat_data, cmap='RdYlGn', aspect='auto',
               vmin=-max(abs(heat_data.min()), abs(heat_data.max())),
               vmax=max(abs(heat_data.min()), abs(heat_data.max())))
ax.set_xticks(range(len(heat_m))); ax.set_xticklabels([LAB[m] for m in heat_m], rotation=20)
ax.set_yticks(range(len(sectors))); ax.set_yticklabels(sectors)
for si in range(len(sectors)):
    for mi in range(len(heat_m)):
        ax.text(mi, si, f'{heat_data[si,mi]:+.1f}%', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, label='Mean Total Return (%)')
ax.set_title('Sector-Level Mean Total Return (%)', fontweight='bold')
plt.tight_layout()
savefig('fig4_sector_heatmap')

# Figure 5: Significance
comp_m = [m for m in ALL_METHODS if m != 'causal_dqn']
cd_vals = [c_ds.get(m, 0)    for m in comp_m]
lp_vals = [-np.log10(corrected.get(m, 1) + 1e-10) for m in comp_m]
colors  = ['#E63946' if corrected.get(m, 1) < 0.05 else '#AAAAAA' for m in comp_m]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(range(len(comp_m)), cd_vals,  color=colors, alpha=0.85)
axes[0].axvline(0, color='gray', lw=0.8); axes[0].axvline(0.5, color='orange', lw=1, ls='--')
axes[0].axvline(0.8, color='red', lw=1, ls='--')
axes[0].set_yticks(range(len(comp_m))); axes[0].set_yticklabels([LAB.get(m,m) for m in comp_m])
axes[0].set_xlabel("Cohen's d"); axes[0].set_title('Effect Size', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
axes[1].barh(range(len(comp_m)), lp_vals, color=colors, alpha=0.85)
axes[1].axvline(-np.log10(0.05), color='orange', lw=1.5, ls='--', label='p=0.05')
axes[1].axvline(-np.log10(0.01), color='red',    lw=1.5, ls='--', label='p=0.01')
axes[1].set_yticks(range(len(comp_m))); axes[1].set_yticklabels([LAB.get(m,m) for m in comp_m])
axes[1].set_xlabel('-log10(Bonferroni p)'); axes[1].set_title('Statistical Significance', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='x')
plt.suptitle('CausalDQN vs All Baselines - Effect Size and Significance', fontsize=12)
plt.tight_layout()
savefig('fig5_significance')

# Figure 6: Causal graph (AAPL)
try:
    import networkx as nx
    demo_g, demo_vn, _ = learn_causal_structure(
        TradingDataLoader('AAPL', TRAIN_START, TRAIN_END).load(), method='pc')
    G   = nx.DiGraph()
    G.add_nodes_from(demo_vn)
    for i in range(len(demo_vn)):
        for j in range(len(demo_vn)):
            if demo_g[i, j] != 0:
                G.add_edge(demo_vn[i], demo_vn[j])
    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.spring_layout(G, seed=42, k=2.5)
    node_colors = ['#E63946' if n == 'action' else
                   '#457B9D' if n == 'returns' else '#A8DADC' for n in G.nodes()]
    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors, node_size=1800,
                     font_size=8, font_weight='bold', arrows=True,
                     arrowsize=20, edge_color='#333333', width=1.5,
                     connectionstyle='arc3,rad=0.1')
    ax.set_title('Learned Causal Graph - AAPL (PC algorithm, alpha=0.05)\n'
                 'Red=action, Blue=returns, Light blue=features', fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    savefig('fig6_causal_graph')
except Exception as e:
    print(f'Fig 6 skipped: {e}')


OK  Saved: fig2_ablation.pdf/.png
OK  Saved: fig3_risk_return.pdf/.png
OK  Saved: fig4_sector_heatmap.pdf/.png
OK  Saved: fig5_significance.pdf/.png
    [PC alpha=0.05] 12 nodes, 19 edges
OK  Saved: fig6_causal_graph.pdf/.png


## 17. Final Summary

In [21]:
cr_ret = np.mean(all_ret['causal_dqn'])   if all_ret['causal_dqn'] else float('nan')
dd_ret = np.mean(all_ret['dueling_dqn'])  if all_ret['dueling_dqn'] else float('nan')
bh_ret = np.mean(all_ret['buy_hold'])     if all_ret['buy_hold'] else float('nan')
cr_sh  = np.mean(all_sharpe['causal_dqn']) if all_sharpe['causal_dqn'] else float('nan')
dd_sh  = np.mean(all_sharpe['dueling_dqn']) if all_sharpe['dueling_dqn'] else float('nan')

best_comp = max([m for m in ALL_METHODS if m != 'causal_dqn' and all_ret.get(m)],
                key=lambda m: np.mean(all_ret[m]))

print('='*70)
print('EXPERT SYSTEMS WITH APPLICATIONS - SUBMISSION CLAIMS')
print('='*70)
print(f"""
Dataset   : {len(valid)} assets | 5 sectors | global markets (US, Asia, EU, Commodities)
Period    : Train {TRAIN_START}-{TRAIN_END} | Val {VAL_START}-{VAL_END} | Test {TEST_START}-{TEST_END}
Trials    : {N_TRIALS} per method (seeds 100-{100+N_TRIALS-1})  |  Episodes: {N_EPISODES}
Device    : {DEVICE}

Performance:
  Causal DQN (ours) : {cr_ret*100:+.2f}%  |  Sharpe {cr_sh:.3f}
  Dueling DQN       : {dd_ret*100:+.2f}%  |  Sharpe {dd_sh:.3f}
  Buy-Hold          : {bh_ret*100:+.2f}%
  Best competitor   : {best_comp} ({np.mean(all_ret[best_comp])*100:+.2f}%)

Improvements:
  vs Dueling DQN : {(cr_ret-dd_ret)/(abs(dd_ret)+1e-9)*100:+.1f}%
  vs Buy-Hold    : {(cr_ret-bh_ret)/(abs(bh_ret)+1e-9)*100:+.1f}%

Statistics (Wilcoxon + Bonferroni):
  vs Dueling DQN : p={corrected.get('dueling_dqn', float('nan')):.6f}  d={c_ds.get('dueling_dqn',0):.3f}
  vs Buy-Hold    : p={corrected.get('buy_hold', float('nan')):.6f}  d={c_ds.get('buy_hold',0):.3f}
  Friedman (RL)  : chi2={f_stat:.3f}  p={f_pval:.6f}

Saved outputs:
  journal_ablation.csv
  journal_full_results.csv
  fig1_distributions.pdf/png  - violin plots
  fig2_ablation.pdf/png       - ablation bar charts
  fig3_risk_return.pdf/png    - risk-return frontier
  fig4_sector_heatmap.pdf/png - sector heatmap
  fig5_significance.pdf/png   - Cohen's d + Bonferroni p
  fig6_causal_graph.pdf/png   - AAPL causal graph
""")


EXPERT SYSTEMS WITH APPLICATIONS - SUBMISSION CLAIMS

Dataset   : 30 assets | 5 sectors | global markets (US, Asia, EU, Commodities)
Period    : Train 2018-01-01-2021-12-31 | Val 2022-01-01-2022-12-31 | Test 2023-01-01-2024-12-31
Trials    : 10 per method (seeds 100-109)  |  Episodes: 100
Device    : cuda

Performance:
  Causal DQN (ours) : +25.59%  |  Sharpe 0.481
  Dueling DQN       : +28.68%  |  Sharpe 0.552
  Buy-Hold          : +78.91%
  Best competitor   : buy_hold (+78.91%)

Improvements:
  vs Dueling DQN : -10.8%
  vs Buy-Hold    : -67.6%

Statistics (Wilcoxon + Bonferroni):
  vs Dueling DQN : p=0.830537  d=-0.056
  vs Buy-Hold    : p=0.000242  d=-0.746
  Friedman (RL)  : chi2=17.157  p=0.008723

Saved outputs:
  journal_ablation.csv
  journal_full_results.csv
  fig1_distributions.pdf/png  - violin plots
  fig2_ablation.pdf/png       - ablation bar charts
  fig3_risk_return.pdf/png    - risk-return frontier
  fig4_sector_heatmap.pdf/png - sector heatmap
  fig5_significance.pdf/